# Model Training
## Telco Customer Churn Prediction

This notebook trains and compares multiple machine learning models for churn prediction.

### Contents:
1. Data Loading
2. Model Comparison
3. Hyperparameter Tuning
4. Customer Segmentation (Clustering)
5. Save Best Models

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings

# Add src to path
sys.path.append('../src')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import joblib

warnings.filterwarnings('ignore')

# Import project modules
from config import (
    TRAIN_DATA_DIR, VAL_DATA_DIR, TEST_DATA_DIR, MODELS_DIR,
    RANDOM_STATE, CV_FOLDS, N_JOBS, create_directories
)
from models.classification import ChurnClassifier, compare_models
from models.clustering import CustomerSegmentation, analyze_segments

create_directories()
print('Libraries imported!')

## 1. Data Loading

In [ ]:
# Load data splits
train = pd.read_csv(TRAIN_DATA_DIR / 'train.csv')
val = pd.read_csv(VAL_DATA_DIR / 'val.csv')
test = pd.read_csv(TEST_DATA_DIR / 'test.csv')

print(f'Train: {len(train)} samples')
print(f'Validation: {len(val)} samples')
print(f'Test: {len(test)} samples')

In [ ]:
# Prepare feature columns
feature_cols = [col for col in train.columns if 
                col.endswith('_scaled') or col.endswith('_encoded') or
                col in ['HighValue', 'ShortTermContract', 'AutomaticPayment', 'TotalServices']]

target_col = 'Churn_Binary'

print(f'Number of features: {len(feature_cols)}')
print(f'Target column: {target_col}')

In [ ]:
# Prepare X and y
X_train = train[feature_cols].values
y_train = train[target_col].values

X_val = val[feature_cols].values
y_val = val[target_col].values

X_test = test[feature_cols].values
y_test = test[target_col].values

print(f'X_train shape: {X_train.shape}')
print(f'y_train distribution: {np.bincount(y_train)}')
print(f'Churn rate: {y_train.mean():.2%}')

## 2. Model Comparison

In [ ]:
# Define models to compare
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100, class_weight='balanced', n_jobs=N_JOBS),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE, n_estimators=100),
    'XGBoost': XGBClassifier(random_state=RANDOM_STATE, n_estimators=100, use_label_encoder=False, eval_metric='logloss')
}

print('Models to compare:')
for name in models:
    print(f'  - {name}')

In [ ]:
# Train and evaluate each model
results = []

for name, model in models.items():
    print(f'\nTraining {name}...')
    
    # Cross-validation on training data
    cv_scores = cross_val_score(model, X_train, y_train, cv=CV_FOLDS, scoring='roc_auc', n_jobs=N_JOBS)
    print(f'  CV ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
    
    # Train on full training set
    model.fit(X_train, y_train)
    
    # Evaluate on validation set
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]
    
    accuracy = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    roc_auc = roc_auc_score(y_val, y_proba)
    
    results.append({
        'Model': name,
        'CV ROC-AUC': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Val Accuracy': accuracy,
        'Val F1': f1,
        'Val ROC-AUC': roc_auc
    })
    
    print(f'  Validation - Accuracy: {accuracy:.4f}, F1: {f1:.4f}, ROC-AUC: {roc_auc:.4f}')

In [ ]:
# Compare results
results_df = pd.DataFrame(results).sort_values('Val ROC-AUC', ascending=False)
print('\nModel Comparison Results:')
print(results_df.to_string(index=False))

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['Val Accuracy', 'Val F1', 'Val ROC-AUC']
colors = ['steelblue', 'orange', 'green', 'red']

for i, metric in enumerate(metrics):
    ax = axes[i]
    sorted_df = results_df.sort_values(metric, ascending=True)
    ax.barh(sorted_df['Model'], sorted_df[metric], color=colors)
    ax.set_xlabel(metric)
    ax.set_title(f'{metric} by Model')
    ax.set_xlim([sorted_df[metric].min() * 0.9, sorted_df[metric].max() * 1.05])
    
    # Add value labels
    for j, v in enumerate(sorted_df[metric]):
        ax.text(v + 0.005, j, f'{v:.3f}', va='center')

plt.tight_layout()
plt.show()

## 3. Hyperparameter Tuning (Best Model)

In [ ]:
# Select best model for tuning (XGBoost typically performs well)
best_model_name = results_df.iloc[0]['Model']
print(f'Best performing model: {best_model_name}')

# Use XGBoost for tuning demonstration
print('\nTuning XGBoost hyperparameters...')

In [ ]:
# Define parameter grid for XGBoost
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

print('Parameter grid:')
for param, values in param_grid.items():
    print(f'  {param}: {values}')

In [ ]:
# Simplified grid search (reduced parameter space for faster execution)
simple_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 7],
    'learning_rate': [0.05, 0.1]
}

xgb_model = XGBClassifier(
    random_state=RANDOM_STATE,
    use_label_encoder=False,
    eval_metric='logloss'
)

grid_search = GridSearchCV(
    xgb_model, 
    simple_param_grid, 
    cv=CV_FOLDS,
    scoring='roc_auc',
    n_jobs=N_JOBS,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV ROC-AUC: {grid_search.best_score_:.4f}')

In [ ]:
# Evaluate best model on validation set
best_xgb = grid_search.best_estimator_

y_pred_val = best_xgb.predict(X_val)
y_proba_val = best_xgb.predict_proba(X_val)[:, 1]

print('\nBest XGBoost - Validation Results:')
print(f'Accuracy: {accuracy_score(y_val, y_pred_val):.4f}')
print(f'F1 Score: {f1_score(y_val, y_pred_val):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_val, y_proba_val):.4f}')

print('\nClassification Report:')
print(classification_report(y_val, y_pred_val, target_names=['No Churn', 'Churn']))

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_xgb.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(10, 8))
top_n = 15
top_features = feature_importance.head(top_n).sort_values('importance')
plt.barh(top_features['feature'], top_features['importance'], color='steelblue')
plt.xlabel('Importance')
plt.title(f'Top {top_n} Feature Importances (XGBoost)')
plt.tight_layout()
plt.show()

print('\nTop 10 Features:')
print(feature_importance.head(10).to_string(index=False))

## 4. Customer Segmentation (Clustering)

In [ ]:
# Prepare data for clustering
cluster_features = ['tenure_scaled', 'MonthlyCharges_scaled', 'TotalCharges_scaled']
X_cluster = train[cluster_features].values

print(f'Clustering features: {cluster_features}')
print(f'Data shape: {X_cluster.shape}')

In [ ]:
# Find optimal number of clusters
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_range = range(2, 10)
inertias = []
silhouettes = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = kmeans.fit_predict(X_cluster)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_cluster, labels))
    print(f'k={k}: inertia={kmeans.inertia_:.2f}, silhouette={silhouettes[-1]:.4f}')

In [ ]:
# Plot elbow and silhouette
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow method
axes[0].plot(k_range, inertias, 'bo-', linewidth=2)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

# Silhouette score
axes[1].plot(k_range, silhouettes, 'go-', linewidth=2)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score by k')

# Mark best k
best_k = list(k_range)[np.argmax(silhouettes)]
axes[1].axvline(x=best_k, color='red', linestyle='--', label=f'Best k={best_k}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nOptimal number of clusters: {best_k}')

In [ ]:
# Train final clustering model
n_clusters = 4  # Use 4 clusters based on analysis

kmeans_final = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=10)
train['Segment'] = kmeans_final.fit_predict(X_cluster)

print(f'\nCluster distribution:')
print(train['Segment'].value_counts().sort_index())

In [ ]:
# Analyze segments
segment_analysis = train.groupby('Segment').agg({
    'tenure': 'mean',
    'MonthlyCharges': 'mean',
    'TotalCharges': 'mean',
    'Churn_Binary': 'mean',
    'customerID': 'count'
}).round(2)

segment_analysis.columns = ['Avg Tenure', 'Avg Monthly Charges', 'Avg Total Charges', 'Churn Rate', 'Size']
segment_analysis['Churn Rate'] = (segment_analysis['Churn Rate'] * 100).round(1).astype(str) + '%'

print('\nCustomer Segments Analysis:')
print(segment_analysis)

In [ ]:
# Visualize segments
fig = plt.figure(figsize=(14, 5))

# Scatter plot
ax1 = fig.add_subplot(121)
scatter = ax1.scatter(train['tenure'], train['MonthlyCharges'], 
                       c=train['Segment'], cmap='viridis', alpha=0.5, s=30)
ax1.set_xlabel('Tenure (months)')
ax1.set_ylabel('Monthly Charges ($)')
ax1.set_title('Customer Segments')
plt.colorbar(scatter, ax=ax1, label='Segment')

# Add cluster centers
centers = kmeans_final.cluster_centers_
# Plot centers (need to reverse scale)

# Churn rate by segment
ax2 = fig.add_subplot(122)
churn_by_segment = train.groupby('Segment')['Churn_Binary'].mean() * 100
colors = ['green' if x < 20 else 'orange' if x < 35 else 'red' for x in churn_by_segment]
ax2.bar(churn_by_segment.index, churn_by_segment.values, color=colors)
ax2.set_xlabel('Segment')
ax2.set_ylabel('Churn Rate (%)')
ax2.set_title('Churn Rate by Segment')

for i, v in enumerate(churn_by_segment.values):
    ax2.text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## 5. Save Best Models

In [ ]:
# Create ChurnClassifier wrapper and save
churn_classifier = ChurnClassifier(model_type='xgboost')
churn_classifier.model = best_xgb
churn_classifier.is_fitted = True
churn_classifier.feature_importances_ = best_xgb.feature_importances_

# Save churn model
churn_classifier.save(MODELS_DIR / 'churn_model.pkl')

In [ ]:
# Save clustering model
clustering_model = CustomerSegmentation('kmeans', n_clusters=n_clusters)
clustering_model.model = kmeans_final
clustering_model.is_fitted = True
clustering_model.labels_ = kmeans_final.labels_
clustering_model.cluster_centers_ = kmeans_final.cluster_centers_

clustering_model.save(MODELS_DIR / 'clustering_model.pkl')

In [ ]:
# Save feature list
feature_info = {
    'feature_columns': feature_cols,
    'cluster_features': cluster_features,
    'n_clusters': n_clusters,
    'best_params': grid_search.best_params_
}

joblib.dump(feature_info, MODELS_DIR / 'feature_info.pkl')
print(f'Saved feature info to {MODELS_DIR / "feature_info.pkl"}')

In [ ]:
# Summary
print('\n' + '='*60)
print('MODEL TRAINING COMPLETE!')
print('='*60)
print(f'''
Best Classification Model: XGBoost
  - Best params: {grid_search.best_params_}
  - CV ROC-AUC: {grid_search.best_score_:.4f}
  - Validation ROC-AUC: {roc_auc_score(y_val, y_proba_val):.4f}

Customer Segmentation:
  - Method: KMeans
  - Number of clusters: {n_clusters}

Models saved to:
  - Churn model: {MODELS_DIR / 'churn_model.pkl'}
  - Clustering model: {MODELS_DIR / 'clustering_model.pkl'}
  - Feature info: {MODELS_DIR / 'feature_info.pkl'}
''')